In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

C:\Users\atliu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Loading Dataset

In [7]:
abstracts_data = pd.read_csv('arxiv_abstracts_final.csv')
df = abstracts_data.copy()

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 182969 entries, 0 to 182968
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   paper_id    182969 non-null  object
 1   title       182969 non-null  object
 2   abstract    182969 non-null  object
 3   published   182969 non-null  object
 4   categories  182969 non-null  object
dtypes: object(5)
memory usage: 7.0+ MB


# Cleaning Dataset

In [9]:
df = (df.drop_duplicates(subset=['paper_id']).assign(
    published=pd.to_datetime(df["published"]), 
    text= lambda x: x["title"] + " " + x["abstract"])
    .sort_values("published")
    .reset_index(drop=True)
)

Data colleced using window search, so it contains some dublicated articles.  For next step, I combined title and abstract.

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179699 entries, 0 to 179698
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   paper_id    179699 non-null  object        
 1   title       179699 non-null  object        
 2   abstract    179699 non-null  object        
 3   published   179699 non-null  datetime64[ns]
 4   categories  179699 non-null  object        
 5   text        179699 non-null  object        
dtypes: datetime64[ns](1), object(5)
memory usage: 8.2+ MB


In [11]:
df.head()

,paper_id,title,abstract,published,categories,text
0,http://arxiv.org/abs/2101.00117v1,Multi-task Retrieval for Knowledge-Intensive T...,Retrieving relevant contexts from a large corp...,2021-01-01,['cs.CL'],Multi-task Retrieval for Knowledge-Intensive T...
1,http://arxiv.org/abs/2101.00204v4,BanglaBERT: Language Model Pretraining and Ben...,"In this work, we introduce BanglaBERT, a BERT-...",2021-01-01,['cs.CL'],BanglaBERT: Language Model Pretraining and Ben...
2,http://arxiv.org/abs/2101.03235v1,Key Phrase Extraction & Applause Prediction,With the increase in content availability over...,2021-01-01,"['cs.CL', 'cs.AI']",Key Phrase Extraction & Applause Prediction Wi...
3,http://arxiv.org/abs/2101.00234v3,Subformer: Exploring Weight Sharing for Parame...,Transformers have shown improved performance w...,2021-01-01,"['cs.CL', 'cs.LG']",Subformer: Exploring Weight Sharing for Parame...
4,http://arxiv.org/abs/2101.00259v2,Code Generation from Natural Language with Les...,Training datasets for semantic parsing are typ...,2021-01-01,"['cs.CL', 'cs.LG']",Code Generation from Natural Language with Les...


In [12]:
df = df[['paper_id', 'published', 'text', 'categories']]

# Text Embedding

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [14]:
embeddings = model.encode(df['text'].tolist(), show_progress_bar=True, batch_size=64)

Batches: 100%|██████████| 2808/2808 [46:00<00:00,  1.02it/s]


In [15]:
np.save("embeddings.npy", embeddings)
df.to_parquet("documents.parquet");